# Lecture 1 — OpenRouter API Testing

**Goal**: Explore different LLM models through OpenRouter API using resume data:

1. Check account credit balance
2. List available models with pricing
3. Compare outputs from Claude vs free/cheap models on resume analysis
4. Understand API request/response patterns
5. Observe differences in model capabilities

We'll use a dataset of 130 candidate resumes throughout this lecture series. Today we focus on API mechanics — using resumes as our test data to compare how different models handle the same tasks.

## Setup
This notebook requires:
- `OPENROUTER_API_KEY` (enter directly in Cell 1)


In [12]:
# Add current directory to Python path for imports
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))


import json
from typing import Any, Dict, List
import pandas as pd

from openrouter_utils import (
    check_credits,
    print_remaining_credits,
    list_models,
    chat_completion,
    safe_chat,
    display_comparison,
    load_resumes,
    load_job_requirements
)

OPENROUTER_API_KEY = "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"  # Paste your key here

# Models to test
MODELS = {
    "claude": "anthropic/claude-sonnet-4.6",
    "qwen-free": "qwen/qwen3.6-plus-preview:free",
    "GPT-5.4 Pro": "openai/gpt-5.4-pro",
    "bytedance seed": "bytedance-seed/seed-2.0-lite",
}

print("Imports loaded")
print(f"Testing {len(MODELS)} models:", list(MODELS.keys()))

if not OPENROUTER_API_KEY or OPENROUTER_API_KEY.strip() == "":
    raise RuntimeError(
        "⚠️  Please set OPENROUTER_API_KEY above before running this notebook.\n"
        "Get your key from: https://openrouter.ai/keys"
    )

print("✓ API key configured")

Imports loaded
Testing 4 models: ['claude', 'qwen-free', 'GPT-5.4 Pro', 'bytedance seed']
✓ API key configured


In [13]:
print_remaining_credits(OPENROUTER_API_KEY)

💳 API Key Credit Balance:
   Key limit:    $10.00
   Key usage:    $0.08
   Remaining:    $9.92


In [14]:
# Execute - function is imported from openrouter_utils.py
models_list = list_models(OPENROUTER_API_KEY, limit=100)

# Parse into DataFrame for easy viewing
models_df = pd.DataFrame([
    {
        "id": m.get("id", ""),
        "name": m.get("name", ""),
        "prompt_cost": m.get("pricing", {}).get("prompt", "N/A"),
        "completion_cost": m.get("pricing", {}).get("completion", "N/A"),
        "context_length": m.get("context_length", "N/A"),
    }
    for m in models_list
])

print(f"Found {len(models_df)} models")
print("\nOur test models:")
for key, model_id in MODELS.items():
    match = models_df[models_df["id"] == model_id]
    if not match.empty:
        row = match.iloc[0]
        print(f"  {key:10s} → ${row['prompt_cost']}/1M prompt tokens")
    else:
        print(f"  {key:10s} → {model_id} (not found in list)")

# Display sample of all models
models_df.head(20)

Found 100 models

Our test models:
  claude     → $0.000003/1M prompt tokens
  qwen-free  → $0/1M prompt tokens
  GPT-5.4 Pro → $0.00003/1M prompt tokens
  bytedance seed → $0.00000025/1M prompt tokens


,id,name,prompt_cost,completion_cost,context_length
0,x-ai/grok-4.20-multi-agent,xAI: Grok 4.20 Multi-Agent,0.000002,0.000006,2000000
1,x-ai/grok-4.20,xAI: Grok 4.20,0.000002,0.000006,2000000
2,google/lyria-3-pro-preview,Google: Lyria 3 Pro Preview,0,0,1048576
3,google/lyria-3-clip-preview,Google: Lyria 3 Clip Preview,0,0,1048576
4,qwen/qwen3.6-plus-preview:free,Qwen: Qwen3.6 Plus Preview (free),0,0,1000000
5,kwaipilot/kat-coder-pro-v2,Kwaipilot: KAT-Coder-Pro V2,0.0000003,0.0000012,256000
6,reka/reka-edge,Reka Edge,0.0000001,0.0000001,16384
7,xiaomi/mimo-v2-omni,Xiaomi: MiMo-V2-Omni,0.0000004,0.000002,262144
8,xiaomi/mimo-v2-pro,Xiaomi: MiMo-V2-Pro,0.000001,0.000003,1048576
9,minimax/minimax-m2.7,MiniMax: MiniMax M2.7,0.0000003,0.0000012,204800


In [15]:
# Load resume data
resumes = load_resumes('../data/resumes_final.csv')
print(f"Loaded {len(resumes)} resumes")

# Pick one sample resume for our API tests
sample_id = list(resumes.keys())[0]
sample_resume = resumes[sample_id]['Resume_str']

print(f"\nSample resume ID: {sample_id}")
print(f"Preview (first 500 chars):\n{sample_resume[:500]}...")

# Define test prompts that showcase model differences using resume data
PROMPTS = [
    {
        "name": "Resume Summary",
        "messages": [
            {
                "role": "user",
                "content": f"Summarize this resume in 2-3 sentences:\n\n{sample_resume}"
            }
        ]
    },
    {
        "name": "Candidate Strengths",
        "messages": [
            {
                "role": "user",
                "content": f"What are this candidate's top 3 strengths? Be specific and cite evidence from the resume.\n\n{sample_resume}"
            }
        ]
    },
    {
        "name": "Experience Level",
        "messages": [
            {
                "role": "user",
                "content": f"Rate this candidate's experience level as one of: junior, mid, or senior. Explain your reasoning in 2-3 sentences.\n\n{sample_resume}"
            }
        ]
    }
]

print(f"\nDefined {len(PROMPTS)} test prompts: {[p['name'] for p in PROMPTS]}")

Loaded 130 resumes

Sample resume ID: 10089434
Preview (first 500 chars):
         INFORMATION TECHNOLOGY TECHNICIAN I       Summary     Versatile Systems Administrator possessing superior troubleshooting skills for networking issues, end user problems, and network security. Experienced in server management, systems analysis, and offering in-depth understanding of IT infrastructure areas. Detail-oriented, independent, and focused on taking a systematic approach to solving complex problems. Demonstrated exceptional technical knowledge and skills while working with vari...

Defined 3 test prompts: ['Resume Summary', 'Candidate Strengths', 'Experience Level']


### Before Going Further

Verify the information above: 
* Where is the resume we are looking at? -- Its the first one in resumes_final.csv under the data file
* Did it get the correct information? -- Yes

In [16]:
# Run all prompts through all models

results = []

for prompt_obj in PROMPTS:
    prompt_name = prompt_obj["name"]
    messages = prompt_obj["messages"]
    
    print(f"\n{'='*60}")
    print(f"Prompt: {prompt_name}")
    print(f"{'='*60}")
    
    for model_key, model_id in MODELS.items():
        print(f"\nTesting {model_key}...")
        
        result = chat_completion(
            OPENROUTER_API_KEY,
            model_id,
            messages,
            temperature=0.7,
            max_tokens=500
        )
        
        results.append({
            "prompt": prompt_name,
            "model_key": model_key,
            "model_id": model_id,
            **result
        })
        
        # Display output
        if result["error"]:
            print(f"  ❌ Error: {result['error']}")
        else:
            content = result["content"]
            preview = content
            print(f"  ✓ Response: {preview}")
            
            usage = result.get("usage", {})
            if usage:
                print(f"    Tokens: {usage.get('prompt_tokens', 0)} prompt + {usage.get('completion_tokens', 0)} completion")

print("\n✓ All tests complete")


Prompt: Resume Summary

Testing claude...
  ✓ Response: This IT professional has over 15 years of experience as a Systems Administrator/IT Technician, with expertise in Microsoft technologies including Active Directory, Office 365, Azure, and Exchange, as well as virtualization, enterprise backup management, and network security. They have a strong background in cloud migration, disaster recovery planning, and scripting using PowerShell and VBScript, along with experience managing both Windows and Linux environments. They hold a Bachelor of Science in Information Technology from Florida International University and a CompTIA Network+ certification.
    Tokens: 1907 prompt + 113 completion

Testing qwen-free...
  ✓ Response: Versatile IT professional with over 15 years of experience in systems administration, network infrastructure, and cloud environments, specializing in Azure, Office 365, and VMware. Expertise spans Active Directory and Group Policy management, enterprise backup and 

### Before going further

* How do we add a different model? -- add more in the MODELS
* Add 2-3 additional models to test the results -- added chatgpt 5.4 pro and bytedance

In [22]:
# Convert results to DataFrame for analysis

results_df = pd.DataFrame(results)

# Add computed fields
results_df["response_length"] = results_df["content"].str.len()
results_df["has_error"] = results_df["error"].notna()
results_df["total_tokens"] = results_df["usage"].apply(
    lambda u: u.get("total_tokens", 0) if isinstance(u, dict) else 0
)

print(f"Collected {len(results_df)} responses")
print(f"Errors: {results_df['has_error'].sum()}")

# Summary by model
summary = results_df.groupby("model_key").agg({
    "has_error": "sum",
    "response_length": "mean",
    "total_tokens": "sum"
}).round(1)

summary.columns = ["Errors", "Avg Response Length", "Total Tokens Used"]
print("\nSummary by Model:")
print(summary)

results_df.head()

Collected 12 responses
Errors: 0

Summary by Model:
                Errors  Avg Response Length  Total Tokens Used
model_key                                                     
GPT-5.4 Pro          0                501.5               6069
bytedance seed       0               1454.3               9757
claude               0               1017.0               6398
qwen-free            0               1172.0               9768


,prompt,model_key,model_id,model,content,parsed_content,usage,error,response_length,has_error,total_tokens
0,Resume Summary,claude,anthropic/claude-sonnet-4.6,anthropic/claude-sonnet-4.6,This IT professional has over 15 years of expe...,None,"{'prompt_tokens': 1907, 'completion_tokens': 1...",None,600.0,False,2020
1,Resume Summary,qwen-free,qwen/qwen3.6-plus-preview:free,qwen/qwen3.6-plus-preview:free,Versatile IT professional with over 15 years o...,None,"{'prompt_tokens': 1732, 'completion_tokens': 9...",None,625.0,False,2688
2,Resume Summary,GPT-5.4 Pro,openai/gpt-5.4-pro,openai/gpt-5.4-pro,Experienced IT professional with nearly two de...,None,"{'prompt_tokens': 1664, 'completion_tokens': 1...",None,557.0,False,1832
3,Resume Summary,bytedance seed,bytedance-seed/seed-2.0-lite,bytedance-seed/seed-2.0-lite,"A versatile, detail-oriented IT professional w...",None,"{'prompt_tokens': 1805, 'completion_tokens': 1...",None,880.0,False,2912
4,Candidate Strengths,claude,anthropic/claude-sonnet-4.6,anthropic/claude-sonnet-4.6,# Top 3 Strengths of This Candidate\n\n## 1. 🖥...,None,"{'prompt_tokens': 1915, 'completion_tokens': 4...",None,1744.0,False,2318


### Before Going Further

What are your take-aways from the table above? -- different response lengths
* How doe the model resposnes compare? Why? -- different tokenization / embedding / training

In [18]:
# Function is now imported from openrouter_utils.py
# Display all comparisons
for prompt_obj in PROMPTS:
    display_comparison(results_df, prompt_obj["name"])


Prompt: Resume Summary

[CLAUDE] (anthropic/claude-sonnet-4.6)
----------------------------------------------------------------------
This IT professional has over 15 years of experience as a Systems Administrator/IT Technician, with expertise in Microsoft technologies including Active Directory, Office 365, Azure, and Exchange, as well as virtualization, enterprise backup management, and network security. They have a strong background in cloud migration, disaster recovery planning, and scripting using PowerShell and VBScript, along with experience managing both Windows and Linux environments. They hold a Bachelor of Science in Information Technology from Florida International University and a CompTIA Network+ certification.

Tokens: 2020 total


[QWEN-FREE] (qwen/qwen3.6-plus-preview:free)
----------------------------------------------------------------------
Versatile IT professional with over 15 years of experience in systems administration, network infrastructure, and cloud enviro

In [19]:
# Structured Output Example: Extracting Resume Information
# This demonstrates extracting structured data from unstructured resume text

# Build the extraction prompt using the same sample resume
extraction_prompt = (
    "Extract key information from the following resume and return it as structured JSON.\n\n"
    f"Resume:\n{sample_resume}\n\n"
    "Extract the following information and return it as a JSON object with this exact structure:\n"
    "{\n"
    '  "candidate_name": "string or null if not found",\n'
    '  "current_title": "string (most recent job title)",\n'
    '  "years_of_experience": "number (estimate total years from work history)",\n'
    '  "education": {\n'
    '    "degree": "string (highest degree)",\n'
    '    "field": "string (field of study)",\n'
    '    "institution": "string (school name)"\n'
    "  },\n"
    '  "technical_skills": ["string (list of technical skills mentioned)"],\n'
    '  "soft_skills": ["string (list of soft/professional skills mentioned)"],\n'
    '  "experience_level": "junior | mid | senior (based on years and responsibilities)",\n'
    '  "industries": ["string (industries the candidate has worked in)"],\n'
    '  "certifications": ["string (any certifications mentioned)"],\n'
    '  "key_achievements": ["string (notable accomplishments, up to 3)"]\n'
    "}\n\n"
    "Important:\n"
    "- If a value is not mentioned in the resume, use null for that field\n"
    "- For lists, return an empty list [] if no items are found\n"
    "- Return ONLY valid JSON, no additional text or markdown formatting"
)

messages = [
    {"role": "user", "content": extraction_prompt}
]

output = {}
for model in MODELS:

    print(f"Running {model}")
    
    result = chat_completion(
        OPENROUTER_API_KEY,
        MODELS[model],
        messages,
        temperature=0.1,  # Very low temperature for consistent, accurate extraction
        max_tokens=800,   # Enough tokens for detailed JSON
        response_format={"type": "json_object"}  # Request JSON mode
    )

    output[model] = result

Running claude
Running qwen-free
Running GPT-5.4 Pro
Running bytedance seed


### Before going further

* What is a structured Response?
* Why will it help us?

In [23]:
for model in output:
    if not output[model].get('parsed_content'):
        print(f"Model {model} has no structured output")
        continue
    
    parsed = output[model]['parsed_content']
    print(f"\n{'='*50}")
    print(f"Model: {model}")
    print(f"{'='*50}")
    print(f"  Name: {parsed.get('candidate_name', 'N/A')}")
    print(f"  Title: {parsed.get('current_title', 'N/A')}")
    print(f"  Experience Level: {parsed.get('experience_level', 'N/A')}")
    print(f"  Years of Experience: {parsed.get('years_of_experience', 'N/A')}")
    skills = parsed.get('technical_skills', [])
    print(f"  Technical Skills ({len(skills)}): {skills[:10]}")
    certs = parsed.get('certifications', [])
    print(f"  Certifications: {certs}")


Model: claude
  Name: None
  Title: Information Technology Technician I
  Experience Level: senior
  Years of Experience: 18
  Technical Skills (56): ['Active Directory', 'Group Policy Objects (GPO)', 'PowerShell', 'VBScript', 'Microsoft Exchange', 'VMware vSphere', 'Office 365', 'Microsoft Azure', 'StorSimple', 'Twinstrata iSCSI']
  Certifications: ['CompTIA Network+ (2014)']

Model: qwen-free
  Name: None
  Title: Information Technology Technician I
  Experience Level: senior
  Years of Experience: 19
  Technical Skills (26): ['Active Directory', 'Group Policy Objects (GPO)', 'PowerShell', 'VBScript', 'Microsoft Exchange', 'VMware vSphere', 'Office 365', 'Microsoft Azure', 'Linux', 'SCCM']
  Certifications: ['CompTIA Network+ (2014)']
Model GPT-5.4 Pro has no structured output

Model: bytedance seed
  Name: None
  Title: Information Technology Technician I
  Experience Level: senior
  Years of Experience: 19
  Technical Skills (70): ['Active Directory', 'Group Policy Objects', 'Powe

## Preview: Job Requirements

In lectures 2-4, you'll use the job description below to **score and route candidates** against these requirements. For now, just read through it to get familiar with the role we'll be hiring for.


In [24]:
# Load and display the job requirements we'll use in future lectures
from IPython.display import Markdown, display

job_req = load_job_requirements('../data/job_req_senior.md')
display(Markdown(job_req))

## About the Role

We are seeking an experienced Full-Stack Software Engineer to join our growing engineering team. The ideal candidate will have a strong background in both frontend and backend development, with expertise in .NET technologies, modern web frameworks, and cloud infrastructure. You will work on building and maintaining enterprise applications that serve thousands of users while collaborating with cross-functional teams in an Agile environment.

## Key Responsibilities

- Design, develop, and maintain full-stack web applications using .NET/C# backend and modern JavaScript frameworks
- Build and optimize RESTful APIs for internal and external consumption
- Work with SQL databases to design schemas, write efficient queries, and optimize performance
- Collaborate with product managers and designers to translate requirements into technical solutions
- Deploy and maintain applications on AWS cloud infrastructure
- Participate in code reviews and contribute to technical documentation
- Mentor junior developers and contribute to team knowledge sharing
- Work in an Agile/Scrum environment with regular sprints and standups

## Required Qualifications

### Technical Skills
- **Backend Development:**
  - 5+ years of experience with C# and .NET Framework/.NET Core
  - Strong proficiency in ASP.NET, ASP.NET MVC, or Web API
  - Experience with Entity Framework, ADO.NET, or similar ORMs

- **Frontend Development:**
  - Proficiency in JavaScript (ES6+), HTML5, and CSS3
  - Experience with at least one modern framework (React, Angular, or Vue)
  - Understanding of responsive design principles

- **Database:**
  - Strong SQL skills (Microsoft SQL Server, MySQL, or PostgreSQL)
  - Experience designing database schemas and writing stored procedures
  - Understanding of database optimization and indexing strategies

- **Version Control & DevOps:**
  - Proficient with Git for version control
  - Experience with CI/CD pipelines (Jenkins, Azure DevOps, or similar)
  - Familiarity with containerization (Docker) is a plus

### Professional Skills
- 5-10 years of professional software development experience
- Bachelor's degree in Computer Science, Information Technology, or related field (or equivalent experience)
- Strong problem-solving and analytical skills
- Excellent communication and collaboration abilities
- Experience working in Agile/Scrum development environments

## Preferred Qualifications

- **Cloud Experience:**
  - AWS experience (EC2, S3, Lambda, RDS)
  - AWS certification (Solutions Architect or Developer) is a plus
  - Experience with Azure or Google Cloud Platform

- **Additional Technologies:**
  - Experience with microservices architecture
  - Knowledge of Linux/Unix systems administration
  - Familiarity with Python or Java
  - Experience with message queues (RabbitMQ, Kafka) or Redis

- **Security & Best Practices:**
  - Understanding of web security principles (OWASP Top 10)
  - Experience with authentication systems (OAuth, JWT, Active Directory)
  - Knowledge of API design best practices (RESTful principles)

- **Tools & Frameworks:**
  - Experience with project management tools (JIRA, Azure DevOps)
  - Familiarity with testing frameworks (xUnit, NUnit, Jest)
  - Knowledge of ORM tools and migration strategies


**Note to Recruiters:** Please do not contact us regarding this position. We work directly with candidates only.

**Application Deadline:** Applications reviewed on a rolling basis

**Job ID:** SWE-FS-2026-001
